## Xây thư viện ML nhỏ bằng NumPy

### Import NumPy và tạo hai mẫu để kiểm tra

In [2]:
import numpy as np
from sympy.codegen.numpy_nodes import logaddexp
from sympy.stats.rv import probability

X_demo = np.array([[2.0, 1.0],
                   [0.0, 1.0]])
y_demo = np.array([1,0])

print ("X_demo.shape:", X_demo.shape)
print("Y_demo.shape:", y_demo.shape)


X_demo.shape: (2, 2)
Y_demo.shape: (2,)


### Sigmoid và BCE

In [3]:
def sigmoid(z):
    return np.exp(-np.logaddexp(0,-z))

def binary_cross_entropy(y, logits):
    losses = np.logaddexp(0, logits) - y * logits
    return losses.mean()
z_demo = np.array([0.0, 0.0])
print("xác suất:", sigmoid(z_demo))
print("BCE:", binary_cross_entropy(y_demo, z_demo))

xác suất: [0.5 0.5]
BCE: 0.6931471805599453


### viết gradient và kiểm tra một bước cập nhật

In [4]:
def logistic_gradient(X,y,p):
    error = p - y
    dw = X.T @ error/len(y)
    db = error.mean()
    return dw, db

w_demo = np.zeros(X_demo.shape[1])
b_demo = 0.0
learning_rate = 0.1

z_before = X_demo @ w_demo + b_demo
p_before = sigmoid(z_before)
dw, db = logistic_gradient(X_demo, y_demo, p_before)

w_demo = w_demo - learning_rate * dw
b_demo = b_demo - learning_rate * db
z_after = X_demo @ w_demo + b_demo

print("dw:", dw, "| db", db)
print("w mới:", w_demo, "| b mới:", b_demo)
print("BCE trước:", binary_cross_entropy(y_demo, z_before))  # ≈ 0.6931
print("BCE sau:", binary_cross_entropy(y_demo, z_after))    # ≈ 0.6688



dw: [-0.5  0. ] | db 0.0
w mới: [0.05 0.  ] | b mới: 0.0
BCE trước: 0.6931471805599453
BCE sau: 0.6687719203167581


### Gom các hàm vào class LogisticRegression

In [5]:
class LogisticRegression:
    def __init__(self, learning_rate=0.1, n_steps=1000):
        self.learning_rate = learning_rate
        self.n_steps = n_steps
    def fit(self,X,y):
        self.w = np.zeros(X.shape[1])
        self.b = 0.0
        self.loss_history = []

        for _ in range(self.n_steps):
            z = X @ self.w +self.b
            p = sigmoid(z)
            self.loss_history.append(binary_cross_entropy(y, z))

            dw, db = logistic_gradient(X,y, p)
            self.w -= self.learning_rate * dw
            self.b -= self.learning_rate * db

        return self

    def predict_proba(self, X):
        return sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        probabilities = self.predict_proba(X)
        return (probabilities >= threshold).astype(int)

### Tạo dữ liệu giả lập và chia train/test

In [6]:
rng = np.random.default_rng(seed=42)
X = rng.normal(size=(500,2))
noise = rng.normal(scale=0.5, size = len(X))
score = 2 * X[:, 0] - X[:, 1] + 0.3 + noise
y = (score >= 0).astype(int)

indices = rng.permutation(len(y))
train_idx = indices[:400]
test_idx = indices[400:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

print("Train:", X_train.shape)  # (400, 2)
print("Test:", X_test.shape)    # (100, 2)

Train: (400, 2)
Test: (100, 2)


### Train và đánh giá bằng NumPy

In [7]:
model = LogisticRegression(learning_rate=0.1, n_steps=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

train_logits = X_train @ model.w + model.b
test_logits = X_test @ model.w + model.b

print("w học được:", model.w.round(3))
print("b học được:", round(model.b, 3))
print("BCE train ban đầu:", round(model.loss_history[0], 4))
print("BCE train sau cập nhật cuối:",
      round(binary_cross_entropy(y_train, train_logits), 4))
print("BCE test:", round(binary_cross_entropy(y_test, test_logits), 4))
print("Accuracy test:", (y_pred == y_test).mean())

w học được: [ 4.178 -2.233]
b học được: 0.601
BCE train ban đầu: 0.6931
BCE train sau cập nhật cuối: 0.2016
BCE test: 0.2138
Accuracy test: 0.91
